In [1]:
import pandas as pd
import numpy as np

import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,f1_score

In [2]:
df = pd.read_csv(r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\data\processed\model_ready_dataset.csv")

df.head()

,HDF,OSF,PWF,TWF,rpm_torque_interaction,Rotational speed [rpm],load_stress,Torque [Nm],load_density,Tool wear [min],temperature_ratio,tool_wear_mean_10,temperature_difference,air_temp_mean_10,UDI,Machine failure
0,0,0,0,0,71177.0,1306,29.7025,54.5,0.545,50,1.034806,36.8,10.4,298.60,19,0
1,0,0,0,0,53040.0,1632,10.5625,32.5,0.325,55,1.034794,40.2,10.4,298.64,20,0
2,0,0,0,0,58712.5,1375,18.2329,42.7,0.427,58,1.034794,43.6,10.4,298.69,21,0
3,0,0,0,0,64960.0,1450,20.0704,44.8,0.448,63,1.035141,47.0,10.5,298.71,22,0
4,0,0,0,0,48536.7,1581,9.4249,30.7,0.307,65,1.034794,50.1,10.4,298.74,23,0


In [3]:
X = df.drop("Machine failure", axis=1)

y = df["Machine failure"]

In [4]:
X.columns = (
    X.columns
    .str.replace("[","",regex=False)
    .str.replace("]","",regex=False)
    .str.replace("{","",regex=False)
    .str.replace("}","",regex=False)
    .str.replace(":","",regex=False)
    .str.replace(",","",regex=False)
)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [6]:
model = joblib.load(r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\models\final_lightgbm_model.pkl")

In [7]:
original_prediction = model.predict(X_test)

original_f1 = f1_score(y_test, original_prediction)

original_f1

np.float64(0.951048951048951)

In [8]:
def add_noise(data, noise_level):
    noisy_data = data.copy()

    noise = np.random.normal(loc=0, scale=noise_level, size=noisy_data.shape)

    noisy_data += noise

    return noisy_data

In [9]:
noise_results = []

for level in [0.01,0.05,0.1]:
    noisy_test = add_noise(X_test, noise_level=level)

    prediction = model.predict(noisy_test)

    noise_results.append({
        "Noise Level": level,
        "F1 Score": f1_score(y_test, prediction)
    })

In [10]:
noise_df = pd.DataFrame(noise_results)

noise_df

,Noise Level,F1 Score
0,0.01,0.071055
1,0.05,0.070796
2,0.10,0.072149


In [11]:
noise_df.to_csv(r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\data\processed\noise_sensitivity_results.csv", index=False)